## Traffic Demand Prediction Model
Loading the dependencies for the task.

In [2]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

### Load the Datasets
Reading `train.csv` and `test.csv`.

In [3]:
train = pd.read_csv("./dataset/train.csv")
test = pd.read_csv("./dataset/test.csv")

print("Train size:", train.shape)
print("Test size:", test.shape)

Train size: (77299, 11)
Test size: (41778, 10)


### Feature Engineering
Parsing time fields, deriving `time_in_mins`, `day_of_week` and setting default categorical/numerical imputations to prevent target leaks.

In [15]:
from sklearn.cluster import KMeans
def preprocess(df):
    df_new = df.copy()
    
    df_new['hour'] = df_new['timestamp'].apply(lambda x: int(x.split(':')[0]))
    df_new['minute'] = df_new['timestamp'].apply(lambda x: int(x.split(':')[1]))
    df_new['time_in_mins'] = df_new['hour'] * 60 + df_new['minute']
    MINUTES_IN_DAY = 1440.0
    df_new['time_sin'] = np.sin(2 * np.pi * df_new['time_in_mins'] / MINUTES_IN_DAY)
    df_new['time_cos'] = np.cos(2 * np.pi * df_new['time_in_mins'] / MINUTES_IN_DAY)
    
    for col in ['RoadType', 'LargeVehicles', 'Landmarks']:
        if col in df_new.columns:
            df_new[col] = df_new[col].fillna('Unknown')
    
    # temp_mean = df_new['Temperature'].mean()
    lanes_mean = df_new['NumberofLanes'].mean()
    df_new['Temperature'] = df_new.groupby('timestamp')['Temperature'].transform(lambda x: x.fillna(x.median()))
    df_new['NumberofLanes'] = df_new['NumberofLanes'].fillna(lanes_mean)
    df_new['day_of_week'] = df_new['day'] % 7
    df_new['is_weekend'] = (df_new['day_of_week'] >= 5).astype(int)
    df_new['temp_bins'] = pd.cut(df_new['Temperature'], bins=range(int(df_new['Temperature'].min())-5, int(df_new['Temperature'].max())+5, 5))
    df_new['Weather'] = df_new.groupby('temp_bins', dropna=False)['Weather'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'Clear')
    )

    # Drop the temporary bin column when done
    df_new = df_new.drop(columns=['temp_bins'])
    import pygeohash as pgh

    # Decode geohash into center latitude and longitude
    df_new['latitude'] = df_new['geohash'].apply(lambda x: pgh.decode(x)[0])
    df_new['longitude'] = df_new['geohash'].apply(lambda x: pgh.decode(x)[1])
    coords = df_new[['latitude', 'longitude']]
    categorical_features = ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather']
    df_new = pd.get_dummies(df_new, columns=categorical_features, drop_first=True, dtype=int)
    return df_new

X_train = preprocess(train)
X_test = preprocess(test)

y_train = X_train['demand']
unused_columns = ['Index', 'geohash', 'day', 'timestamp', 'hour', 'minute', 'demand']
X_train = X_train.drop(columns = unused_columns, axis=1)

X_test_ids = X_test['Index']
unused_columns = ['Index', 'geohash', 'day', 'timestamp', 'hour', 'minute']
X_test = X_test.drop(columns = unused_columns, axis=1)

/var/folders/9x/3x7xy3b937b7s21dnh9vzpp00000gn/T/ipykernel_24160/652344098.py:23: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_new['Weather'] = df_new.groupby('temp_bins', dropna=False)['Weather'].transform(
/var/folders/9x/3x7xy3b937b7s21dnh9vzpp00000gn/T/ipykernel_24160/652344098.py:23: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_new['Weather'] = df_new.groupby('temp_bins', dropna=False)['Weather'].transform(


### K-Fold Cross Validation Setup
Training CatBoost using 5 independent folds to maximize evaluation `R2` metrics.

In [27]:
# categorical_features = ['RoadType', 'LargeVehicles', 'Landmarks', 'Weather']

# kf = KFold(n_splits=5, shuffle=True, random_state=42)
# test_preds = np.zeros(len(X_test))
# oof_preds = np.zeros(len(X_train))

# print("Starting 5-Fold Training...")
# for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
#     print(f"--- Fold {fold+1} ---")
#     X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
#     X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]
    
#     model = CatBoostRegressor(
#         iterations=4000,
#         learning_rate=0.06,
#         depth=8,
#         l2_leaf_reg=4,
#         loss_function='RMSE',
#         eval_metric='R2',
#         random_seed=42 + fold,
#         early_stopping_rounds=150,
#         verbose=1000
#     )
    
#     model.fit(
#         X_tr, y_tr,
#         eval_set=(X_val, y_val),
#         cat_features=categorical_features,
#         use_best_model=True
#     )
    
#     oof_preds[val_idx] = model.predict(X_val)
#     test_preds += model.predict(X_test) / kf.n_splits

# oof_r2 = r2_score(y_train, oof_preds)
# print(f"\nOverall Out-Of-Fold R2 Score: {oof_r2:.6f}")
# import xgboost as xgb
# kf = KFold(n_splits=5, shuffle=True, random_state=42)

# # Arrays to store out-of-fold and test predictions
# oof_preds = np.zeros(len(X_train))
# test_preds = np.zeros(len(X_test))

# print("Starting XGBoost 5-Fold Training (Without Lags)...")

# for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
#     print(f"\n--- Fold {fold+1} ---")
    
#     # Split the encoded data
#     X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
#     X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]
    
#     # 5. Initialize XGBoost Regressor
#     model = xgb.XGBRegressor(
#         n_estimators=4000,
#         learning_rate=0.06,
#         max_depth=8,                  # Matches your CatBoost tree depth
#         reg_lambda=4,                 # L2 regularization parameter
#         objective='reg:squarederror',
#         eval_metric='rmse',
#         random_state=42 + fold,
#         tree_method='hist',           # Enabled for ultra-fast training speeds
#         n_jobs=-1
#     )
    
#     # 6. Fit the model using validation set for early stopping
#     model.fit(
#         X_tr, y_tr,
#         eval_set=[(X_val, y_val)],
#         early_stopping_rounds=150,     # Stops if validation error doesn't drop for 150 rounds
#         verbose=1000                  # Prints evaluation metrics every 1000 trees
#     )
    
#     # 7. Record Out-Of-Fold and Test Predictions
#     oof_preds[val_idx] = model.predict(X_val)
#     test_preds += model.predict(X_test_encoded) / kf.n_splits

# # 8. Calculate and Print Overall Out-Of-Fold R2 Score
# oof_r2 = r2_score(y_train, oof_preds)
# print(f"\nOverall Out-Of-Fold R2 Score: {oof_r2:.6f}")

import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import numpy as np

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Arrays to store out-of-fold and test predictions
oof_preds = np.zeros(len(X_train))
test_preds = np.zeros(len(X_test))

print("Starting XGBoost 5-Fold Training (Without Lags)...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
    print(f"\n--- Fold {fold+1} ---")
    
    # Split the encoded data
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]
    
    # 5. Initialize XGBoost Regressor (Early stopping moved here)
    model = xgb.XGBRegressor(
        n_estimators=4000,
        learning_rate=0.06,
        max_depth=10,                  # Matches your CatBoost tree depth
        reg_lambda=4,                 # L2 regularization parameter
        objective='reg:squarederror',
        eval_metric='rmse',
        early_stopping_rounds=150,     # <--- FIXED: Moved here to prevent error
        random_state=42 + fold,
        tree_method='hist',           # Enabled for ultra-fast training speeds
        n_jobs=-1
    )
    
    # 6. Fit the model cleanly without the deprecated argument
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=1000                  # Prints evaluation metrics every 1000 trees
    )
    
    # 7. Record Out-Of-Fold and Test Predictions
    oof_preds[val_idx] = model.predict(X_val)
    test_preds += model.predict(X_test) / kf.n_splits

# 8. Calculate and Print Overall Out-Of-Fold R2 Score
oof_r2 = r2_score(y_train, oof_preds)
print(f"\nOverall Out-Of-Fold R2 Score: {oof_r2:.6f}")

Starting XGBoost 5-Fold Training (Without Lags)...

--- Fold 1 ---
[0]	validation_0-rmse:0.13513
[1000]	validation_0-rmse:0.03261
[1657]	validation_0-rmse:0.03249

--- Fold 2 ---
[0]	validation_0-rmse:0.13788
[1000]	validation_0-rmse:0.03489
[1599]	validation_0-rmse:0.03482

--- Fold 3 ---
[0]	validation_0-rmse:0.13505
[1000]	validation_0-rmse:0.03232
[1301]	validation_0-rmse:0.03229

--- Fold 4 ---
[0]	validation_0-rmse:0.13164
[1000]	validation_0-rmse:0.03414
[1432]	validation_0-rmse:0.03404

--- Fold 5 ---
[0]	validation_0-rmse:0.13607
[1000]	validation_0-rmse:0.03292
[1535]	validation_0-rmse:0.03281

Overall Out-Of-Fold R2 Score: 0.945177


### Generate Submission File
Clipping variables to limits `(0, None)` as demand can't be negative, and saving.

In [6]:
submission = pd.DataFrame({
    'Index': X_test_ids,
    'demand': np.clip(test_preds, 0, None)
})

submission.to_csv("submission.csv", index=False)
print("Submission saved!")

Submission saved!
